# CfC -- Closed-form Continuous-time Networks

Hasani et al., *Closed-form Continuous-time Neural Networks*, Nature MI 2022 ([arXiv:2106.13898](https://arxiv.org/abs/2106.13898)).

`h(t) = sigma(-f(x)*t) * g(x) + (1 - sigma(-f(x)*t)) * h(x)` -- a closed-form approximation of the LTC ODE's solution, so no ODE solver is needed at rollout time. See `model.py` and `../../papers/README.md`.

This notebook trains a `CfCModel` on the UCI Person Activity dataset (the same one used in the original CfC/NCP papers).

In [ ]:
import sys
sys.path.insert(0, '../..')
sys.path.insert(0, '.')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from liquid_playground.data import load_person_activity
from liquid_playground.device import resolve_device
from liquid_playground.utils.seed import set_seed
from model import CfCModel

set_seed(0)
device = resolve_device('auto')  # or 'cpu' / 'cuda' / 'mps'
print('device:', device)

In [ ]:
train_x, train_y, test_x, test_y = load_person_activity()
train_x, test_x = train_x.to(device), test_x.to(device)
train_y, test_y = train_y.to(device), test_y.to(device)
n_classes = int(train_y.max().item()) + 1
print(train_x.shape, 'n_classes =', n_classes)

In [ ]:
model = CfCModel(input_size=train_x.shape[-1], hidden_size=64, output_size=n_classes).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'test_acc': []}
epochs = 40
for epoch in range(epochs):
    model.train()
    opt.zero_grad()
    logits = model(train_x)
    loss = loss_fn(logits, train_y)
    loss.backward()
    opt.step()

    model.eval()
    with torch.no_grad():
        test_acc = (model(test_x).argmax(-1) == test_y).float().mean().item()
    history['train_loss'].append(loss.item())
    history['test_acc'].append(test_acc)

print(f"final test accuracy: {history['test_acc'][-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history['train_loss']); axes[0].set_title('train loss'); axes[0].set_xlabel('epoch')
axes[1].plot(history['test_acc']); axes[1].set_title('test accuracy'); axes[1].set_xlabel('epoch')
fig.tight_layout()
plt.show()